# 📘 RAG básico con LlamaIndex sobre documentos de dominio específico

Esta notebook muestra cómo construir un índice vectorial a partir de la resolución de creación del LICDIA (UNLu).

👉 Ejecutá todo paso a paso en Google Colab.

In [ ]:
# ✅ Instalación de dependencias
!pip install llama-index langchain-openai --quiet

## 📥 Descargar el PDF del Plan de Estudios

Descargamos el pdf automáticamente desde un dominio unlu-edu.ar.

In [ ]:
import gdown

gdown.download(id="1AddV4FMq70bOO4REUgAVppe5i4Vw9h_W", output="documento.pdf", quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1AddV4FMq70bOO4REUgAVppe5i4Vw9h_W
To: /content/documento.pdf
100%|██████████| 1.35M/1.35M [00:00<00:00, 138MB/s]


'documento.pdf'

## 📄 Cargar y dividir el libro en nodos

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
import os

reader = SimpleDirectoryReader(input_files=["documento.pdf"])
documents = reader.load_data()

splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents)

print(f"Se generaron {len(nodes)} nodos.")

Se generaron 6 nodos.


## 🔎 Crear índice y hacer consultas

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-..."  # reemplazá por tu clave

In [ ]:
from llama_index.core import VectorStoreIndex

# Crear índice vectorial
index = VectorStoreIndex(nodes)
query_engine = index.as_query_engine(similarity_top_k=3)

In [ ]:
consulta = "¿Qué es el LICDIA?"

In [ ]:
from langchain_openai import ChatOpenAI

# 2. Consulta directa al modelo sin contexto
llm = ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo")
response = llm.invoke(consulta)
print("💬 Sin RAG:\n", response.content)

💬 Sin RAG:
 El LICDIA es el acrónimo de "Licenciatura en Ciencias de la Información y Documentación", una carrera universitaria que forma profesionales especializados en la gestión de la información y la documentación en diferentes ámbitos, como bibliotecas, archivos, centros de documentación, empresas, instituciones públicas, entre otros. Los graduados en LICDIA están capacitados para organizar, clasificar, preservar y difundir la información de manera eficiente y efectiva.


In [ ]:
# 1. Consulta usando RAG
response_rag = query_engine.query(consulta)

print("🔍 Con RAG:\n", response_rag)

🔍 Con RAG:
 El LICDIA es el Laboratorio de Investigación en Ciencias de Datos e Inteligencia Artificial de la Universidad Nacional de Luján.
